In [1]:
import pandas as pd
import numpy as np

In [2]:
train = pd.read_csv("../data/raw/train.csv")
store = pd.read_csv("../data/raw/store.csv")

print("Train shape:", train.shape)
print("Store shape:", store.shape)

Train shape: (1017209, 9)
Store shape: (1115, 10)


C:\Users\aarya\AppData\Local\Temp\ipykernel_21552\4246006486.py:1: DtypeWarning: Columns (7: StateHoliday) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv("../data/raw/train.csv")


In [3]:
train["Date"] = pd.to_datetime(train["Date"])

print("Date range:")
print(train["Date"].min(), "to", train["Date"].max())

Date range:
2013-01-01 00:00:00 to 2015-07-31 00:00:00


In [4]:
daily = train.merge(
    store,
    on="Store",
    how="left"
)

print("Merged shape:", daily.shape)

Merged shape: (1017209, 18)


In [5]:
daily = daily.sort_values(
    ["Store", "Date"]
).reset_index(drop=True)

print(daily[["Store", "Date", "Sales"]].head(10))

   Store       Date  Sales
0      1 2013-01-01      0
1      1 2013-01-02   5530
2      1 2013-01-03   4327
3      1 2013-01-04   4486
4      1 2013-01-05   4997
5      1 2013-01-06      0
6      1 2013-01-07   7176
7      1 2013-01-08   5580
8      1 2013-01-09   5471
9      1 2013-01-10   4892


In [6]:
for lag in [1, 7, 14, 30]:
    daily[f"Sales_Lag_{lag}"] = (
        daily.groupby("Store")["Sales"]
        .shift(lag)
    )

print(
    daily[
        [
            "Store",
            "Date",
            "Sales",
            "Sales_Lag_1",
            "Sales_Lag_7",
            "Sales_Lag_14",
            "Sales_Lag_30"
        ]
    ].head(35)
)

    Store       Date  Sales  Sales_Lag_1  Sales_Lag_7  Sales_Lag_14  \
0       1 2013-01-01      0          NaN          NaN           NaN   
1       1 2013-01-02   5530          0.0          NaN           NaN   
2       1 2013-01-03   4327       5530.0          NaN           NaN   
3       1 2013-01-04   4486       4327.0          NaN           NaN   
4       1 2013-01-05   4997       4486.0          NaN           NaN   
5       1 2013-01-06      0       4997.0          NaN           NaN   
6       1 2013-01-07   7176          0.0          NaN           NaN   
7       1 2013-01-08   5580       7176.0          0.0           NaN   
8       1 2013-01-09   5471       5580.0       5530.0           NaN   
9       1 2013-01-10   4892       5471.0       4327.0           NaN   
10      1 2013-01-11   4881       4892.0       4486.0           NaN   
11      1 2013-01-12   4952       4881.0       4997.0           NaN   
12      1 2013-01-13      0       4952.0          0.0           NaN   
13    

In [7]:
sales_shifted = (
    daily.groupby("Store")["Sales"]
    .shift(1)
)

daily["Sales_Rolling_7"] = (
    sales_shifted
    .groupby(daily["Store"])
    .transform(lambda x: x.rolling(7).mean())
)

daily["Sales_Rolling_14"] = (
    sales_shifted
    .groupby(daily["Store"])
    .transform(lambda x: x.rolling(14).mean())
)

daily["Sales_Rolling_30"] = (
    sales_shifted
    .groupby(daily["Store"])
    .transform(lambda x: x.rolling(30).mean())
)

daily["Sales_Rolling_7_Std"] = (
    sales_shifted
    .groupby(daily["Store"])
    .transform(lambda x: x.rolling(7).std())
)

print(
    daily[
        [
            "Store",
            "Date",
            "Sales",
            "Sales_Lag_1",
            "Sales_Lag_7",
            "Sales_Rolling_7",
            "Sales_Rolling_14",
            "Sales_Rolling_30",
            "Sales_Rolling_7_Std"
        ]
    ].head(35)
)

    Store       Date  Sales  Sales_Lag_1  Sales_Lag_7  Sales_Rolling_7  \
0       1 2013-01-01      0          NaN          NaN              NaN   
1       1 2013-01-02   5530          0.0          NaN              NaN   
2       1 2013-01-03   4327       5530.0          NaN              NaN   
3       1 2013-01-04   4486       4327.0          NaN              NaN   
4       1 2013-01-05   4997       4486.0          NaN              NaN   
5       1 2013-01-06      0       4997.0          NaN              NaN   
6       1 2013-01-07   7176          0.0          NaN              NaN   
7       1 2013-01-08   5580       7176.0          0.0      3788.000000   
8       1 2013-01-09   5471       5580.0       5530.0      4585.142857   
9       1 2013-01-10   4892       5471.0       4327.0      4576.714286   
10      1 2013-01-11   4881       4892.0       4486.0      4657.428571   
11      1 2013-01-12   4952       4881.0       4997.0      4713.857143   
12      1 2013-01-13      0       4952

In [8]:
daily["Year"] = daily["Date"].dt.year
daily["Month"] = daily["Date"].dt.month
daily["WeekOfYear"] = daily["Date"].dt.isocalendar().week.astype(int)
daily["IsWeekend"] = (
    daily["Date"].dt.dayofweek >= 5
).astype(int)

print(
    daily[
        [
            "Date",
            "Year",
            "Month",
            "DayOfWeek",
            "WeekOfYear",
            "IsWeekend"
        ]
    ].head(10)
)

        Date  Year  Month  DayOfWeek  WeekOfYear  IsWeekend
0 2013-01-01  2013      1          2           1          0
1 2013-01-02  2013      1          3           1          0
2 2013-01-03  2013      1          4           1          0
3 2013-01-04  2013      1          5           1          0
4 2013-01-05  2013      1          6           1          1
5 2013-01-06  2013      1          7           1          1
6 2013-01-07  2013      1          1           2          0
7 2013-01-08  2013      1          2           2          0
8 2013-01-09  2013      1          3           2          0
9 2013-01-10  2013      1          4           2          0


In [9]:
print("Total rows:", len(daily))

print("\nMissing values in engineered features:")

engineered_cols = [
    "Sales_Lag_1",
    "Sales_Lag_7",
    "Sales_Lag_14",
    "Sales_Lag_30",
    "Sales_Rolling_7",
    "Sales_Rolling_14",
    "Sales_Rolling_30",
    "Sales_Rolling_7_Std"
]

print(daily[engineered_cols].isnull().sum())

Total rows: 1017209

Missing values in engineered features:
Sales_Lag_1             1115
Sales_Lag_7             7805
Sales_Lag_14           15610
Sales_Lag_30           33450
Sales_Rolling_7         7805
Sales_Rolling_14       15610
Sales_Rolling_30       33450
Sales_Rolling_7_Std     7805
dtype: int64


In [10]:
original_rows = len(daily)

daily = daily.dropna(
    subset=[
        "Sales_Lag_1",
        "Sales_Lag_7",
        "Sales_Lag_14",
        "Sales_Lag_30",
        "Sales_Rolling_7",
        "Sales_Rolling_14",
        "Sales_Rolling_30",
        "Sales_Rolling_7_Std"
    ]
).reset_index(drop=True)

removed_rows = original_rows - len(daily)

print("Original rows:", original_rows)
print("Removed rows:", removed_rows)
print("Remaining rows:", len(daily))

print("\nRemaining missing values:")
print(daily.isnull().sum().sum())

Original rows: 1017209
Removed rows: 33450
Remaining rows: 983759

Remaining missing values:
2103141


In [11]:
missing = daily.isnull().sum()

missing = missing[missing > 0].sort_values(ascending=False)

print("Columns with missing values:")
print(missing)

print("\nTotal missing values:", missing.sum())

Columns with missing values:
Promo2SinceYear              491711
Promo2SinceWeek              491711
PromoInterval                491711
CompetitionOpenSinceMonth    312728
CompetitionOpenSinceYear     312728
CompetitionDistance            2552
dtype: int64

Total missing values: 2103141


In [12]:
model_features = [
    "Store",
    "DayOfWeek",
    "Month",
    "Year",
    "WeekOfYear",
    "IsWeekend",
    "Promo",
    "SchoolHoliday",
    "StateHoliday",
    "StoreType",
    "Assortment",
    "CompetitionDistance",
    "Sales_Lag_1",
    "Sales_Lag_7",
    "Sales_Lag_14",
    "Sales_Lag_30",
    "Sales_Rolling_7",
    "Sales_Rolling_14",
    "Sales_Rolling_30",
    "Sales_Rolling_7_Std"
]

target = "Sales"

print("Number of model features:", len(model_features))
print("\nModel features:")
print(model_features)

Number of model features: 20

Model features:
['Store', 'DayOfWeek', 'Month', 'Year', 'WeekOfYear', 'IsWeekend', 'Promo', 'SchoolHoliday', 'StateHoliday', 'StoreType', 'Assortment', 'CompetitionDistance', 'Sales_Lag_1', 'Sales_Lag_7', 'Sales_Lag_14', 'Sales_Lag_30', 'Sales_Rolling_7', 'Sales_Rolling_14', 'Sales_Rolling_30', 'Sales_Rolling_7_Std']


In [13]:
print("Model features:", len(model_features))
print("Target:", target)
print("Total model columns:", len(model_features) + 1)

Model features: 20
Target: Sales
Total model columns: 21


In [14]:
daily["CompetitionDistance"] = daily["CompetitionDistance"].fillna(
    daily["CompetitionDistance"].median()
)

print(
    "Missing CompetitionDistance:",
    daily["CompetitionDistance"].isnull().sum()
)

Missing CompetitionDistance: 0


In [15]:
final_columns = ["Date"] + model_features + [target]

model_df = daily[final_columns].copy()

print("Model shape:", model_df.shape)
print("\nMissing values:", model_df.isnull().sum().sum())
print("\nColumns:")
print(model_df.columns.tolist())

Model shape: (983759, 22)

Missing values: 0

Columns:
['Date', 'Store', 'DayOfWeek', 'Month', 'Year', 'WeekOfYear', 'IsWeekend', 'Promo', 'SchoolHoliday', 'StateHoliday', 'StoreType', 'Assortment', 'CompetitionDistance', 'Sales_Lag_1', 'Sales_Lag_7', 'Sales_Lag_14', 'Sales_Lag_30', 'Sales_Rolling_7', 'Sales_Rolling_14', 'Sales_Rolling_30', 'Sales_Rolling_7_Std', 'Sales']


In [16]:
print("Date range:")
print(model_df["Date"].min(), "to", model_df["Date"].max())

print("\nShape:")
print(model_df.shape)

print("\nMissing values:")
print(model_df.isnull().sum().sum())

print("\nDuplicate rows:")
print(model_df.duplicated().sum())

print("\nTarget statistics:")
print(model_df["Sales"].describe())

Date range:
2013-01-31 00:00:00 to 2015-07-31 00:00:00

Shape:
(983759, 22)

Missing values:
0

Duplicate rows:
0

Target statistics:
count    983759.000000
mean       5794.216983
std        3860.599343
min           0.000000
25%        3745.000000
50%        5765.000000
75%        7881.000000
max       41551.000000
Name: Sales, dtype: float64


In [17]:
import os

os.makedirs("../data/processed", exist_ok=True)

model_df.to_csv(
    "../data/processed/model_data.csv",
    index=False
)

print("Processed dataset saved successfully.")
print("Rows:", len(model_df))
print("Columns:", len(model_df.columns))

Processed dataset saved successfully.
Rows: 983759
Columns: 22


In [18]:
processed_check = pd.read_csv(
    "../data/processed/model_data.csv"
)

print("Saved file shape:", processed_check.shape)
print("Missing values:", processed_check.isnull().sum().sum())

Saved file shape: (983759, 22)
Missing values: 0
